In [ ]:
# 1. Imports
import numpy as np
import pandas as pd

from src.physics.cosmology import Cosmology
from src.physics.symbolic_cosmology import SymbolicCosmology
from src.likelihoods.planck_shoes_joint import PlanckSH0ESJointLikelihood
from src.likelihoods.desi_bao import DESIBAO
from src.likelihoods.cosmic_chronometers import CosmicChronometers
from src.likelihoods.joint_likelihood import JointLikelihood
from src.physics.mcmc_joint_pipeline import JointMCMCPipeline
from src.visualization.paper_figures import (
    plot_hubble_diagram, plot_Hz, plot_residuals, plot_corner
)

# 2. Load data
planck = "data/planck/planck_distance_priors.csv"
bao = "data/desi/desi_bao.csv"
cc = "data/cc/cosmic_chronometers.csv"

planck_shoes = PlanckSH0ESJointLikelihood(planck)
desi_bao = DESIBAO(bao)
cc_like = CosmicChronometers(cc)

# 3. Build joint likelihood
joint = JointLikelihood(planck_shoes, desi_bao, cc_like)

# 4. Define ΛCDM
lcdm = Cosmology("H0*sqrt(Ωm*(1+z)**3 + ΩΛ)", {"H0": 67.4, "Ωm": 0.315, "ΩΛ": 0.685})

# 5. Define S.T.A.R. Model
H_expr = "H0*sqrt(Ωm*(1+z)**3 + ΩΛ + a*z + b*z**2)"
param_names = ["H0", "Ωm", "ΩΛ", "a", "b"]
priors = {
    "H0": (70, 5),
    "Ωm": (0.3, 0.1),
    "ΩΛ": (0.7, 0.1),
    "a": (0, 0.1),
    "b": (0, 0.1),
}
proposal_widths = {k: 0.01 for k in param_names}

# 6. Run MCMC
pipeline = JointMCMCPipeline(H_expr, param_names, priors, proposal_widths, joint)
chain = pipeline.run(theta0=[70, 0.3, 0.7, 0, 0], nsteps=5000)

# 7. Posterior plots
plot_corner(chain, param_names)

# 8. Best-fit model
best_params = dict(zip(param_names, chain.mean(axis=0)))
star = SymbolicCosmology(H_expr, best_params)

# 9. Paper-ready figures
df_planck = pd.read_csv(planck)
plot_hubble_diagram(df_planck, lcdm, star)
plot_Hz([lcdm, star], ["ΛCDM", "S.T.A.R."])
plot_residuals(df_planck, lcdm, star)
